# 第8回 星を、似たものどうしでまとめる


---


## 前回からの接続


第7回は、ネットショップの訪問から購入を見分け、2つの誤りの重さでしきい値を決めました。第6・7回のデータには、当てたい答えの列（迷惑か、購入か）がありました。

今日のデータには、答えの列がありません。答えの列が無いまま、似た行どうしをまとめて群に分けます。


---


## 今日の分析目標


答えの列が無い星を、似たものどうしでまとめると、星の種類が分かれるか。何個に分ければよいかに答える。

まず、星の視差から距離と本当の明るさ（絶対等級）を出し、色と並べて図を描きます。次に、k-meansで星を2つに分け、目で見える星の種類と比べます。そして、k-meansの中で何をくり返しているかを読み、なぜその分け方になったかを確かめます。最後に、群の数を替えて、群の数の指針を並べます。

もう1つのねらいは、止まる条件を書いて、`while`で「動かなくなるまで」くり返す手順を読めるようになることです。k-meansの中身を`while`で書いたものを、1行ずつ読みます。お手本ごとに、コードを1行ずつ読み（「1行ずつ読む」）、変数の中身を見ます（「中身を見る」）。


### 公開データの世界

ヨーロッパ宇宙機関（ESA）の衛星Gaiaは、空の星の位置・明るさ・色を測り続けています。測った値は、誰でも使える公開データとして配られています。今日は、太陽のまわりの星を、そこから取ってきました。


### 場面

夜空の星は、どれも同じに見えます。でも、色と本当の明るさを並べると、星の種類が分かれて見えます。天文学者は、その図を見て、星の種類を見分けてきました。

答えの列（この星は何の種類か）が無いまま、星を似たものどうしでまとめたい。まとめた群が、天文学者の見分ける種類と合うかを確かめます。

似た行どうしをまとめて、いくつかの群に分けることを、クラスタリング（clustering）といいます。群のことをクラスターともいいます。答えの列は使わず、データの値の近さだけで分けます。今日は、いちばん基本の方法、k-meansを使います。


### 手元のデータ

Gaia第3版（DR3）から、太陽の近くの星を、測りの質の条件で絞って取りました。1行が星1つで、27,860個あります。列は、視差（`parallax`）・見かけの等級（`phot_g_mean_mag`）・色（`bp_rp`）の3本と、星の番号（`source_id`）です。答えの列はありません。

データはESAのGaiaアーカイブで公開されていて（CC BY-SA 3.0 IGO）、出典（Gaia Collaboration, Vallenari et al., 2023）とESAへのクレジットを書けば使えます。


### 最後に出す答え

星を似たものどうしでまとめると、星の種類が分かれるか。分かれないなら、なぜか。そして、何個に分ければよいかです。


### 今日の一周（結果だけ先に）

コードの説明はあとにして、今日の解析を、結果だけで一度通して見ておきます。

1. 星を知る：色と本当の明るさを並べると（HR図）、長い帯の主系列と、左下の小さな帯の白色矮星が見えます。目で見ると2種類です。
2. 2つに分ける：えっ、k-meansで2つに分けると、白色矮星が真っ二つになります。主系列も、途中で輪切りになります。
3. くり返しを読む：k-meansは、近い中心の群に入れ、中心を群の平均へ動かすことを、動かなくなるまでくり返します。中心までの距離の二乗和を小さくするので、見た目どおりの分け方より、輪切りのほうを選びます。
4. 群の数を決める：シルエット（silhouette）という指針では、3つがいちばん良くなります。別の指針Gapは、8まで伸び続けます。
5. 出発点を替える：出発点を替えると、止まる所が変わります。いちばん小さい二乗和に届くとは限りません。

このあと、この5つの段の中身を、1つずつ開けていきます。


### 今日の問いとやること

| 段 | 問い | やること |
|---|---|---|
| 星を知る（1節） | 星の何を比べる？ | 視差から距離と本当の明るさを出し、色と並べてHR図を描く |
| 2つに分ける（2節） | 2つに分けると？ | 標準化してk-meansで2つに分け、図に塗り、目安の線と比べる |
| くり返しを読む（3節） | 何をくり返している？ | `while`で書いたk-meansを1行ずつ読み、二乗和を比べる |
| 群の数を決める（4節） | 何個に分ければよい？ | 群の数を替えて、二乗和・Gap・シルエットを並べる |
| 出発点を替える（5節） | 答えは1つか？ | 出発点を20通り替えて、止まった二乗和を並べる |


### 今日の進め方

段ごとに、まずスライドで、その段の結論までを通して聞きます。段の終わりにノートへ移り、その段の節のセルを上から実行して、スライドと同じ数が出るかを確かめます。そして、段ごとに1つあるTODOに答えます。

TODOは、実行の前に結果を予想して書き、1か所を替えて実行し、予想と比べる形です。予想が外れても、かまいません。


---


## 学習ゴール


この回を終えると、次のことができるようになります。

- 視差から距離と絶対等級を出し、HR図を描いて、星の種類を読める
- 列を標準化（standardization）してから、k-meansで行を群に分け、群を図に塗って読める
- k-meansの割り当てと更新を、`while`で「動かなくなるまで」くり返すコードとして読める
- k-meansが距離の二乗和を小さくするので、丸い群を探し、細長い群は輪切りにすることを説明できる
- 群の数を替えて、二乗和・Gap・シルエットを並べ、指針の限界を踏まえて群の数を決められる
- k-meansが出発点しだいで違う所に止まること（局所解）と、`n_init`で何通りも試す理由を説明できる


---


## 0. 準備


### 最初に実行する：道具を呼び出す

- 役割：このノートで使う道具（ライブラリ）を呼び出す
- 入力：なし
- 出力：なし（終わるとセルの左に ✓ が付く）


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


### 図の日本語を表示できるようにする

- 役割：図の中の日本語が □ にならないようにし、図の大きさと字の大きさをそろえる
- 入力：なし
- 出力：「セットアップ完了」の表示

`try:`から先は、日本語表示の道具が動かなくなったときの保険で、中身は読まなくてかまいません。


In [ ]:
# 日本語が図の中で □ になるのを防ぐ
!pip install -q japanize-matplotlib
try:
    import japanize_matplotlib
# japanize が動かなくなったときの保険：同梱フォントを直接登録する
except Exception:
    from importlib.util import find_spec
    from matplotlib import font_manager as fm
    from pathlib import Path
    spec = find_spec('japanize_matplotlib')
    ttf = None
    if spec:
        ttf = next(Path(spec.origin).parent.rglob('*.ttf'), None)
    if ttf:
        fm.fontManager.addfont(str(ttf))
        name = fm.FontProperties(fname=str(ttf)).get_name()
        plt.rcParams['font.family'] = name

plt.rcParams['figure.figsize'] = (9, 5)
plt.rcParams['font.size'] = 13
plt.rcParams['axes.unicode_minus'] = False  # マイナス記号の化けを防ぐ
print('セットアップ完了')


---


## 1. 星の何を比べる？


ここから、段1「星を知る」（1節）です。問いは、星の何を比べればよいか、です。

まず、表を読み込み、3本の列の意味を読みます。


### お手本：表を読み込む

- 役割：星の表を読み込み、星の数を数えて、先頭3行を見る
- 入力：データの置き場所`DATA_DIR`と、ファイル名`gaia_nearby.csv`
- 出力：表`raw`（27,860個）


In [ ]:
# データはこのリポジトリから読み込む
DATA_DIR = ('https://raw.githubusercontent.com/k0heiun0/'
            'applied_exercise_preview/main/shared/data')
raw = pd.read_csv(f'{DATA_DIR}/gaia_nearby.csv')
print(len(raw), '個')
raw.head(3)


#### 1行ずつ読む

- `DATA_DIR`の行：データの置き場所を`DATA_DIR`にしまう（Colabでは、置き場所のURLが2行に分かれて入っている）
- `pd.read_csv`の行：置き場所の`gaia_nearby.csv`を読み、表`raw`にしまう
- `print`の行と最後の行：星の数を表示し、先頭3行を見る


### 3本の列

| 列 | 意味 |
|---|---|
| `parallax` | 視差（ミリ秒角）。大きいほど近い |
| `phot_g_mean_mag` | 見かけの等級。地球から見た明るさ。小さいほど明るい |
| `bp_rp` | 色。青で測った等級と赤で測った等級の差。小さいほど青く、熱い |

地球は1年かけて太陽のまわりを回ります。半年たつと、地球の位置が変わるので、近い星ほど、遠くの星に対して大きくずれて見えます。このずれの角度が視差です。視差から、星までの距離が分かります。


### お手本：列の名前を日本語に

- 役割：英語の列の名前を、日本語の名前に替える
- 入力：表`raw`と、名前の組の辞書`names`
- 出力：表`df`


In [ ]:
names = {'parallax': '視差', 'phot_g_mean_mag': '見かけの等級',
         'bp_rp': '色'}
df = raw.rename(columns=names)
df.head(3)


#### 1行ずつ読む

- 1・2行目：元の名前を鍵に、日本語の名前を値にした辞書を作る（第5回）
- 3行目：辞書のとおりに列の名前を替え、表`df`にしまう
- 4行目：先頭3行を見る


### 見かけの明るさと、本当の明るさ

同じ明るさの星でも、遠くにあれば暗く見えます。星そのものの明るさを比べるには、距離をそろえる必要があります。星を10パーセクの距離に並べたときの等級を、絶対等級といいます。

視差$\varpi$（ミリ秒角）から、距離$d$（パーセク）を出します。

$$
d = \frac{1000}{\varpi}
$$

見かけの等級$m$と距離$d$から、絶対等級$M$を出します。

$$
M = m - 5 \log_{10} \frac{d}{10}
$$


### お手本：距離と絶対等級

- 役割：式のとおりに、距離と絶対等級の列を作る
- 入力：表`df`の視差と見かけの等級
- 出力：表`df`に足した列「距離」「絶対等級」


In [ ]:
df['距離'] = 1000 / df['視差']
df['絶対等級'] = df['見かけの等級'] - 5 * np.log10(df['距離'] / 10)
print(df['距離'].max().round(1))
df[['視差', '距離', '見かけの等級', '絶対等級']].head(3).round(2)


#### 1行ずつ読む

- 1行目：1000を視差で割って、距離の列を作る
- 2行目：式のとおりに、絶対等級の列を作る（`np.log10`は常用対数）
- 3・4行目：いちばん遠い星の距離と、先頭3行を見る

いちばん遠い星でも、50パーセクです。太陽のまわりの、ごく近い星だけを取りました。


### お手本：HR図を描く

- 役割：横に色、縦に絶対等級をとって、星を点で打つ
- 入力：表`df`の色と絶対等級
- 出力：図（HR図）


In [ ]:
plt.figure(figsize=(7, 5))
plt.scatter(df['色'], df['絶対等級'], s=1)
plt.gca().invert_yaxis()
plt.xlabel('色（BP−RP）')
plt.ylabel('絶対等級')
plt.show()


#### 1行ずつ読む

- 1行目：図の大きさを決める
- 2行目：横に色、縦に絶対等級をとり、星を1つずつ点で打つ（`s=1`は小さな点）
- 3行目：縦軸を上下逆にする。等級は小さいほど明るいので、明るい星を上にする
- 4〜6行目：軸の名前を付けて、図を表示する


### HR図を読む

この図を、ヘルツシュプルング・ラッセル図（HR図）といいます。

- 主系列：右下から左上へのびる長い帯。太陽の仲間で、中心で水素を燃やして光る。青く熱い星ほど明るい
- 白色矮星：左下の小さな帯。燃え尽きた星の芯。小さいので、熱く青くても暗い
- 巨星：右上の小さな塊。年老いてふくらんだ星。赤いのに明るい。この近さには少ない


### お手本：白色矮星を数える

- 役割：図を見て、2つの帯のあいだに直線を引き、線より下（暗い側）を白色矮星とする
- 入力：表`df`の色と絶対等級
- 出力：列「白色矮星」（白色矮星なら1）と、その数


In [ ]:
df['白色矮星'] = (df['絶対等級'] > 4 * df['色'] + 8.5).astype(int)
print(df['白色矮星'].sum(), '個')


#### 1行ずつ読む

- 1行目：絶対等級が「4×色＋8.5」より大きい（暗い）星を1、それ以外を0にした列を作る
- 2行目：1の数、つまり白色矮星の数を表示する

この列は、図を見て決めた答えのようなものです。k-meansには渡しません。分けたあとの答え合わせだけに使います。k-meansが使うのは、色と絶対等級の2本だけです。


### 結果を見る前に、予想を書く

星を2つに分けたら、主系列と白色矮星に分かれるでしょうか。目で見ると、2つの帯ははっきり離れています。予想と、その理由を1行で書いてください。


（ここに予想を書く）


### TODO①：距離の条件を替える（予想 → 実行 → 比べる）

下のセルは、視差が20より大きい星だけを取り出して、数を表示します。視差20より大きい星は、50パーセク以内の星です。

まず、予想を書いてください。40にすると、25パーセク以内の星になります。半分の距離にすると、星の数は何個になるでしょうか。


（ここに予想を書く）


予想を書いたら、セルの`20`を`40`に替えて実行してください。


In [ ]:
# TODO: 20 を 40 に替えて実行する
near = df[df['視差'] > 20]
print(len(near), '個')


### やってみよう：予想と比べる

実行した結果を、予想と比べてください。星の数は、半分になりましたか。


（ここに答えを書く）


27,860個が3,421個になり、約8分の1（0.123）になりました。半分ではありません。星は球の中に散らばっています。球の体積は半径の3乗に比例するので、半径を半分にすると、体積は$(1/2)^3 = 1/8$になります。


In [ ]:
print(len(df), '個 →', len(df[df['視差'] > 40]), '個')
print(round(len(df[df['視差'] > 40]) / len(df), 3))


### ここまでで分かったこと

| 段 | 分かったこと |
|---|---|
| 星を知る（1節） | 27,860個の星。HR図に主系列と白色矮星（2,027個）が見える |
| 2つに分ける（2節） | ？（このあとの段で） |
| くり返しを読む（3節） | ？（このあとの段で） |
| 群の数を決める（4節） | ？（このあとの段で） |
| 出発点を替える（5節） | ？（このあとの段で） |

まだ分からないのは、k-meansで分けると、この2つの帯に分かれるか、です。


---


## 2. 2つに分けると？


ここから、段2「2つに分ける」（2節）です。問いは、k-meansで2つに分けると、主系列と白色矮星に分かれるか、です。

視差から距離と絶対等級を出し、色と並べてHR図を描きました。目で見ると、長い帯の主系列と、左下の白色矮星（2,027個）の2種類です。ここから、値の近さだけで、星を2つに分けます。


### 近さは、距離で測る

2つの星の近さを、図の上の距離で測ります。色と絶対等級の差を、それぞれ2乗して足したもの（距離の2乗）です。単位がばらばらだと、幅の大きい列の差ばかりが効いてしまいます。


In [ ]:
print(df[['色', '絶対等級']].std().round(2))


絶対等級の散らばり（標準偏差）は、色の約3倍です。このままだと、明るさの差ばかりで分けてしまいます。


### お手本：標準化する

- 役割：2本の列を、平均0・標準偏差（standard deviation）1にそろえる（第7回のおさらい）
- 入力：表`df`の色と絶対等級
- 出力：配列`X`（星の数×2）


In [ ]:
from sklearn.preprocessing import StandardScaler
X = StandardScaler().fit_transform(df[['色', '絶対等級']])
print(X[:3].round(2))


#### 1行ずつ読む

- 1行目：標準化の道具を呼び出す
- 2行目：2本の列を、平均0・標準偏差1にそろえ、配列`X`にしまう
- 3行目：先頭3つの星の、標準化した値を見る。1列目が色、2列目が絶対等級


### k-means

群の数$k$を先に決めます。今日は、まず$k=2$です。中心を$k$個置き、星をいちばん近い中心の群に入れます。中心の置き方は、道具`KMeans`が決めます。中身は3節で読みます。


### お手本：2つに分ける

- 役割：標準化した2本の列で、星を2つの群に分ける
- 入力：配列`X`と、群の数2
- 出力：列「群」（0か1）と、群ごとの星の数


In [ ]:
from sklearn.cluster import KMeans
km = KMeans(2, n_init=10, random_state=0)
km.fit(X)
df['群'] = km.labels_
print(df['群'].value_counts())


#### 1行ずつ読む

- 1行目：k-meansの道具`KMeans`を呼び出す
- 2行目：群の数2で準備する。`n_init=10`は出発点を10通り試す設定、`random_state`は乱数（random number）のシード
- 3行目：`X`に当てはめて、2つに分ける
- 4行目：星ごとの群の番号（0か1）を`km.labels_`から取り、列「群」にする
- 5行目：群ごとの星の数を数える


#### 中身を見る：群の番号


In [ ]:
df[['色', '絶対等級', '群']].head(3).round(2)


星1つに、群の番号が1つ付きました。番号の0と1は、名前にすぎません。


### お手本：群で塗る

- 役割：HR図の点を、群の番号ごとに色を変えて打つ
- 入力：表`df`の色・絶対等級・群
- 出力：図


In [ ]:
plt.figure(figsize=(7, 5))
plt.scatter(df['色'], df['絶対等級'], s=1, c=df['群'],
            cmap='coolwarm')
plt.gca().invert_yaxis()
plt.xlabel('色（BP−RP）')
plt.ylabel('絶対等級')
plt.show()


#### 1行ずつ読む

- 2・3行目：1節のHR図と同じ点を打ち、`c=df['群']`で、群の番号ごとに色を変える（`cmap`は色の組）
- ほかの行は、1節のHR図と同じ

えっ、白色矮星の帯が、途中で切られました。主系列も、真ん中で切られています。


### お手本：目安の線と比べる

- 役割：目安の線（白色矮星か）と、k-meansの群を、組ごとに数える
- 入力：列「白色矮星」と列「群」
- 出力：2×2の表


In [ ]:
pd.crosstab(df['白色矮星'], df['群'])


#### 1行ずつ読む

- 1行目：目安の線の0・1と、群の0・1の組み合わせごとに、星の数を数える（`pd.crosstab`、第7回）

白色矮星2,027個のうち、1,422個が群0、605個が群1に入りました。白色矮星は2つの群に割れ、どちらの群にも、主系列と白色矮星がまざっています。目で見てはっきり離れた2つの帯を、k-meansは2つに分けませんでした。


> 重要：k-meansの群の番号は、名前にすぎない。番号が何の群かは、分けたあとに中身を見て読む。


### TODO②：群の数を替える（予想 → 実行 → 比べる）

下のセルは、k-meansで星を分け、目安の線と組ごとに数えます。

まず、予想を書いてください。3つに分けると、白色矮星は1つの群にまとまるでしょうか。


（ここに予想を書く）


予想を書いたら、セルの`2`を`3`に替えて実行してください。


In [ ]:
# TODO: 2 を 3 に替えて実行する
km_x = KMeans(2, n_init=10, random_state=0)
km_x.fit(X)
pd.crosstab(df['白色矮星'], km_x.labels_)


### やってみよう：予想と比べる

実行した結果を、予想と比べてください。白色矮星は、1つの群にまとまりましたか。主系列はどうなりましたか。


（ここに答えを書く）


3つにすると、白色矮星2,027個は1つの群にまとまります。かわりに、主系列が2つの群に切られます。目で見える2種類に、群の数をいくつ足しても、主系列は切られ続けます。


### ここまでで分かったこと

| 段 | 分かったこと |
|---|---|
| 星を知る（1節） | 27,860個の星。HR図に主系列と白色矮星（2,027個）が見える |
| 2つに分ける（2節） | k-meansで2つに分けると、白色矮星が1,422個と605個に割れる |
| くり返しを読む（3節） | ？（このあとの段で） |
| 群の数を決める（4節） | ？（このあとの段で） |
| 出発点を替える（5節） | ？（このあとの段で） |

まだ分からないのは、k-meansが中で何をしていて、なぜ輪切りにしたか、です。


---


## 3. 何をくり返している？


ここから、段3「くり返しを読む」（3節）です。問いは、k-meansは何をくり返していて、なぜ輪切りにしたのか、です。

k-meansで2つに分けると、白色矮星が2つの群に割れました。3つに分けると白色矮星はまとまりますが、主系列が切られます。ここから、`KMeans`の中で何をしているかを読みます。


### k-meansの手順

1. 中心を2つ置く（出発点）
2. 星を、いちばん近い中心の群に入れる（割り当て）
3. 中心を、群の星の平均へ動かす（更新）

2と3を、中心が動かなくなるまでくり返します。

星$i$の値を$x_i$、群$j$の中心を$\mu_j$とします。

- 割り当て：$c_i = \arg\min_j \|x_i - \mu_j\|^2$（いちばん近い中心の番号）
- 更新：$\mu_j = \dfrac{1}{n_j} \sum_{c_i = j} x_i$（群$j$の$n_j$個の星の平均）


### 何回くり返せばよい？

中心が動かなくなるまでです。それが何回目かは、やってみるまで分かりません。`for`は、回数を先に決めてくり返します（`range(3)`なら3回）。回数が先に決まらないくり返しには、`while`を使います。


### while：最小の例

- 役割：`x`を2倍し続け、100以上になったら止める
- 入力：なし
- 出力：止まったときの`x`


In [ ]:
x = 1
while x < 100:
    x = x * 2
print(x)


#### 1行ずつ読む

- 1行目：`x`を1にする
- 2行目：「`x`が100より小さいあいだ」、字下げした行をくり返す
- 3行目：`x`を2倍にする
- 4行目：くり返しが終わったら、`x`を表示する


#### 中身を見る：xの動き


In [ ]:
x = 1
while x < 100:
    x = x * 2
    print(x)


128になったところで、条件「100より小さい」を満たさなくなり、止まります。


### お手本：近い中心の群に入れる

- 役割：割り当て（手順2）を、関数`assign`にする
- 入力：星`X`と、2つの中心`C`
- 出力：星ごとの群の番号（0か1）


In [ ]:
def assign(X, C):
    d0 = ((X - C[0]) ** 2).sum(axis=1)
    d1 = ((X - C[1]) ** 2).sum(axis=1)
    return (d1 < d0).astype(int)


#### 1行ずつ読む

- 1行目：関数`assign`を作る。入力は星`X`と中心`C`
- 2行目：星ごとに、中心0までの距離の2乗を出す（色と絶対等級の差を2乗して、横に足す）
- 3行目：同じく、中心1までの距離の2乗を出す
- 4行目：中心1のほうが近ければ1、そうでなければ0を返す


### お手本：中心を平均へ動かす

- 役割：更新（手順3）を、関数`update`にする
- 入力：星`X`と、群の番号`lab`
- 出力：新しい2つの中心


In [ ]:
def update(X, lab):
    c0 = X[lab == 0].mean(axis=0)
    c1 = X[lab == 1].mean(axis=0)
    return np.array([c0, c1])


#### 1行ずつ読む

- 1行目：関数`update`を作る。入力は星`X`と群の番号`lab`
- 2行目：群0の星だけを取り出し、列ごとの平均（色の平均、絶対等級の平均）を出す
- 3行目：群1も同じ
- 4行目：2つの中心を、1つの配列にまとめて返す


### お手本：動かなくなるまでくり返す

- 役割：割り当てと更新を、中心が動かなくなるまでくり返す
- 入力：星`X`と、出発点（0番目と1番目の星）
- 出力：止まるまでの回数`n`、群の番号`lab`、中心`C`


In [ ]:
C = X[[0, 1]]
n = 0
moved = 1.0
while moved > 0:
    lab = assign(X, C)
    C_new = update(X, lab)
    moved = np.abs(C_new - C).max()
    C = C_new
    n = n + 1
print(n, '回で止まった')


#### 1行ずつ読む

- 1行目：出発点。0番目と1番目の星を、2つの中心にする
- 2・3行目：回数`n`を0に、中心の動き`moved`を仮に1にする（最初の1回を必ず回すため）
- 4行目：「中心が少しでも動いたあいだ」、字下げした行をくり返す
- 5・6行目：割り当てて、新しい中心を出す
- 7行目：新しい中心と古い中心の差のうち、いちばん大きいものを`moved`にする
- 8・9行目：中心を入れ替え、回数を1増やす
- 10行目：止まったら、回数を表示する


#### 中身を見る：最初の3回

くり返しを3回だけ回し、中心の動きと、中心までの距離の二乗和を表示します。名前は`C2`のように替え、上の`C`や`lab`を壊さないようにします。


In [ ]:
C2 = X[[0, 1]]
for r in range(3):
    lab2 = assign(X, C2)
    C2_new = update(X, lab2)
    moved2 = np.abs(C2_new - C2).max()
    ss2 = ((X - C2_new[lab2]) ** 2).sum()
    print(r + 1, moved2.round(3), ss2.round(1))
    C2 = C2_new


中心の動きは小さくなっていきます。右の数は、中心までの距離の二乗和（群内平方和）です。二乗和は、割り当てでも更新でも増えません。だから、いつかは動かなくなって止まります。


### whileがしていること

毎回、中身の前に条件（`moved > 0`）を確かめます。満たしていれば、中身を1回（割り当て → 平均へ → `moved`を測る）行い、条件に戻ります。満たさなくなったら、くり返しを抜けて、次の行へ進みます。


In [ ]:
# @title 図：whileの1回ごとの動き（中身は読み飛ばしてよい）
C3 = X[[0, 1]]
moved3 = 1.0
r = 0
rows = []
while moved3 > 0:
    lab3 = assign(X, C3)
    C3_new = update(X, lab3)
    moved3 = np.abs(C3_new - C3).max()
    C3 = C3_new
    r = r + 1
    rows.append([r, moved3, ((X - C3[lab3]) ** 2).sum()])
rows = pd.DataFrame(rows, columns=['回', '中心の動き', '二乗和'])
plt.figure(figsize=(7, 4))
plt.plot(rows['回'], rows['二乗和'], 'o-')
plt.xlabel('くり返した回数')
plt.ylabel('距離の二乗和')
plt.title('条件 moved > 0 を満たすあいだ、くり返す')
plt.show()
print(rows.round(3).tail(3))


> 今日の重要概念（Python）：while
>
> whileは、条件を満たすあいだ、くり返す。何回で終わるかが先に決まらないくり返しに使う。


### お手本：KMeansと比べる

- 役割：同じ出発点から`KMeans`を当てはめ、自分でくり返した答えと比べる
- 入力：配列`X`と、出発点
- 出力：くり返した回数と、群の番号がすべて同じか


In [ ]:
km_s = KMeans(2, init=X[[0, 1]], n_init=1, tol=0)
km_s.fit(X)
print(km_s.n_iter_, (km_s.labels_ == lab).all())


#### 1行ずつ読む

- 1行目：出発点を0番目と1番目の星にし（`init`）、出発点は1通りだけ（`n_init=1`）、止まる条件はぴったり動かなくなるまで（`tol=0`）
- 2行目：当てはめる
- 3行目：くり返した回数と、群の番号がすべて同じか（`all`）を表示する

同じ出発点から始めると、`KMeans`も同じ回数で止まり、群の番号もすべて同じです。`KMeans`の中でしていることは、上の`while`と同じです。


### お手本：見た目どおりから始める

では、目安の線の分け方（主系列と白色矮星）から始めたら、k-meansはどう動くでしょうか。

- 役割：目安の線の群で中心を出し、もう一度割り当てる
- 入力：配列`X`と、列「白色矮星」
- 出力：群を移った星の数


In [ ]:
eye = df['白色矮星'].values
C_eye = update(X, eye)
lab_eye = assign(X, C_eye)
print((lab_eye != eye).sum(), '個が群を移る')


#### 1行ずつ読む

- 1行目：目安の線の0・1を、群の番号として取り出す
- 2・3行目：その群で中心を出し（更新）、もう一度割り当てる
- 4行目：番号が変わった星の数を表示する

3個だけです。見た目どおりの分け方も、k-meansの手順ではほとんど動かず、そこで止まります。止まる先は、出発点しだいで1つではありません。では、`KMeans`は、なぜ輪切りのほうを選んだのでしょうか。


### お手本：二乗和を比べる

- 役割：分け方ごとに、中心までの距離の二乗和を出して比べる
- 入力：配列`X`と、群の番号
- 出力：見た目どおりの分け方と、k-meansの分け方の二乗和


In [ ]:
def ss(X, lab):
    C = update(X, lab)
    return round(((X - C[lab]) ** 2).sum())

print(ss(X, eye), ss(X, lab))


#### 1行ずつ読む

- 1行目：関数`ss`を作る。入力は星`X`と群の番号`lab`
- 2行目：群ごとの中心を出す
- 3行目：`C[lab]`は星ごとの「自分の群の中心」。差を2乗して全部足し、整数に丸めて返す
- 5行目：見た目どおりの分け方と、k-meansの分け方で比べる

見た目どおりの分け方は45,877、k-meansの輪切りは24,035です。輪切りのほうが、二乗和が2倍近く小さいのです。`n_init=10`は、10通りの出発点から始め、二乗和がいちばん小さい答えを選びます。だから、`KMeans`は輪切りを選びました。

主系列は長い帯です。中心を1つにすると、帯の端の星が中心から遠くなります。帯を半分に切れば、どの星も中心に近くなり、二乗和は大きく下がります。


In [ ]:
# @title 図：2つの分け方と、その中心（中心は×。中身は読み飛ばしてよい）
mu = df[['色', '絶対等級']].mean().values
sd = df[['色', '絶対等級']].std(ddof=0).values
fig, axes = plt.subplots(1, 2, figsize=(11, 4), sharey=True)
for ax, g, t in [(axes[0], eye, '見た目どおり'), (axes[1], lab, 'k-means（輪切り）')]:
    ax.scatter(df['色'], df['絶対等級'], s=1, c=g, cmap='coolwarm')
    cc = update(X, g) * sd + mu
    ax.scatter(cc[:, 0], cc[:, 1], s=250, marker='X', color='black')
    ax.set_title(f'{t}：二乗和 {ss(X, g)}')
    ax.set_xlabel('色（BP−RP）')
axes[0].invert_yaxis()
axes[0].set_ylabel('絶対等級')
plt.show()


> 今日の重要概念（統計）：k-meansは丸い群を探す
>
> k-meansは、中心までの距離の二乗和を小さくする分け方を探す。だから丸くまとまった群を好み、細長い群は輪切りにする。


### TODO③：出発点を替える（予想 → 実行 → 比べる）

下のセルは、出発点の2つの星から、割り当てと更新を動かなくなるまでくり返し、目安の線と組ごとに数えます。番号6の星は、白色矮星です。

まず、予想を書いてください。白色矮星の星を出発点にすれば、白色矮星の群ができるでしょうか。


（ここに予想を書く）


予想を書いたら、セルの`[0, 1]`を`[0, 6]`に替えて実行してください。


In [ ]:
# TODO: [0, 1] を [0, 6] に替えて実行する（6番の星は白色矮星）
C = X[[0, 1]]
moved = 1.0
while moved > 0:
    lab_x = assign(X, C)
    C_new = update(X, lab_x)
    moved = np.abs(C_new - C).max()
    C = C_new
pd.crosstab(df['白色矮星'], lab_x)


### やってみよう：予想と比べる

実行した結果を、予想と比べてください。白色矮星の群はできましたか。


（ここに答えを書く）


白色矮星から始めても、白色矮星の群はできません。白色矮星は2つの群に割れ、止まった先は、ほぼ同じ輪切りです。下のセルで、二乗和も同じ24,035になることを確かめられます。出発点の星が白色矮星でも、1回目の更新で、中心は帯の平均へ引っぱられます。


In [ ]:
print(ss(X, lab_x))


### ここまでで分かったこと

| 段 | 分かったこと |
|---|---|
| 星を知る（1節） | 27,860個の星。HR図に主系列と白色矮星（2,027個）が見える |
| 2つに分ける（2節） | k-meansで2つに分けると、白色矮星が1,422個と605個に割れる |
| くり返しを読む（3節） | 割り当てと平均を17回くり返して止まる。二乗和は輪切りが小さい |
| 群の数を決める（4節） | ？（このあとの段で） |
| 出発点を替える（5節） | ？（このあとの段で） |

まだ分からないのは、群の数をいくつにすればよいか、です。


---


## 4. 何個に分ければよい？


ここから、段4「群の数を決める」（4節）です。問いは、群の数は何個にすればよいか、です。

k-meansは、割り当てと更新を、中心が動かなくなるまでくり返します。中心までの距離の二乗和を小さくするので、長い主系列は輪切りになります。ここまで、群の数は2や3と、こちらで決めて渡してきました。

k-meansは、群の数を渡さないと動きません。答えの列が無いので、正解率のような物差しも使えません。データから、群の数の目安を出す指針がいくつかあります。今日は3つ並べます。


### お手本：群の数ごとの二乗和

- 役割：群の数を1から8まで替えて、二乗和をリストにためる
- 入力：配列`X`
- 出力：リスト`ws`（8つの二乗和）


In [ ]:
ws = []
for k in range(1, 9):
    km_k = KMeans(k, n_init=10, random_state=0)
    km_k.fit(X)
    ws.append(km_k.inertia_)
print(np.round(ws, 1))


#### 1行ずつ読む

- 1行目：空のリスト`ws`を用意する（第4回）
- 2〜4行目：群の数`k`を1から8まで替えて、k-meansを当てはめる
- 5行目：二乗和（`inertia_`）をリストの後ろに足す
- 6行目：8つの二乗和を表示する


### お手本：二乗和を描く


In [ ]:
plt.plot(range(1, 9), ws, 'o-')
plt.xlabel('群の数')
plt.ylabel('距離の二乗和')
plt.show()


群を増やせば、どの星も中心に近くなるので、二乗和は必ず下がります。下がり方が急にゆるむ「ひじ」の数を選ぶ考え（エルボー法）があります。でも、この曲線では、ひじがはっきりしません。2か、3か、4か、決めにくいのです。


### Gap統計の考え

群の無いデータでも、群を増やせば二乗和は下がります。そこで、群の無い偽のデータを作り、同じようにk-meansをかけて比べます。本物の二乗和が、偽より大きく下がった分だけ、本物には群があると考えます（Gap統計）。

偽の星は、本物と同じ範囲に、一様（uniform）にばらまいて作ります。帯も塊もありません。


### お手本：偽の星を作る

- 役割：本物と同じ範囲に、同じ数の偽の星を一様にばらまく
- 入力：配列`X`（列ごとの最小と最大、形）
- 出力：配列`fake`


In [ ]:
rng = np.random.default_rng(0)
fake = rng.uniform(X.min(axis=0), X.max(axis=0), size=X.shape)
print(fake[:3].round(2))


#### 1行ずつ読む

- 1行目：乱数を作る道具を、シード0で用意する
- 2行目：列ごとの最小から最大までの範囲に、本物と同じ数の星を一様にばらまく
- 3行目：先頭3つの偽の星を見る


In [ ]:
# @title 図：本物の星と偽の星（中身は読み飛ばしてよい）
fig, axes = plt.subplots(1, 2, figsize=(11, 4), sharey=True)
axes[0].scatter(X[:, 0], X[:, 1], s=1)
axes[0].set_title('本物の星（標準化）')
axes[1].scatter(fake[:, 0], fake[:, 1], s=1, color='gray')
axes[1].set_title('偽の星（同じ範囲に一様にばらまく）')
axes[0].invert_yaxis()
plt.show()


### 式：Gap

群の数$k$の本物の二乗和を$W_k$、偽の二乗和を$W_k^{*}$とします。

$$
\mathrm{Gap}(k) = \overline{\log W_k^{*}} - \log W_k
$$

偽は5組作り、$\log W_k^{*}$を平均します（上の線）。Gapが、次の群の数でもう伸びなくなる、いちばん小さい$k$を選びます。


### お手本：二乗和の対数を返す関数

本物にも偽にも同じ手順を当てるので、関数にします（第7回）。

- 役割：表`Z`を群の数`k`で分け、二乗和の対数を返す
- 入力：配列`Z`と群の数`k`
- 出力：二乗和の対数


In [ ]:
def log_w(Z, k):
    km_z = KMeans(k, n_init=10, random_state=0)
    km_z.fit(Z)
    return np.log(km_z.inertia_)


#### 1行ずつ読む

- 1行目：関数`log_w`を作る。入力は配列`Z`と群の数`k`
- 2・3行目：群の数`k`で、`Z`にk-meansを当てはめる
- 4行目：二乗和の対数（`np.log`は自然対数）を返す


### お手本：群の数ごとのGap

- 役割：群の数を1から8まで替え、偽の星5組と本物で二乗和の対数を出して、Gapをためる
- 入力：配列`X`、関数`log_w`
- 出力：リスト`gaps`（8つのGap）

少し時間がかかります（数十秒）。


In [ ]:
rng = np.random.default_rng(0)
gaps = []
for k in range(1, 9):
    fake_w = []
    for b in range(5):
        fake = rng.uniform(X.min(axis=0), X.max(axis=0), size=X.shape)
        fake_w.append(log_w(fake, k))
    gaps.append(np.mean(fake_w) - log_w(X, k))
print(np.round(gaps, 3))


#### 1行ずつ読む

- 1・2行目：乱数の道具をシード0で用意し直し、空のリスト`gaps`を作る
- 3行目：群の数`k`を1から8まで替える
- 4行目：その`k`での、偽の二乗和の対数をためるリストを作る
- 5〜7行目：偽の星を5組作り、それぞれの二乗和の対数をためる
- 8行目：偽の平均から本物を引いたGapを、`gaps`に足す
- 9行目：8つのGapを表示する


In [ ]:
plt.plot(range(1, 9), gaps, 'o-', color='tab:red')
plt.xlabel('群の数')
plt.ylabel('Gap')
plt.show()


えっ、Gapは、群を増やすたびに伸び続けます。8までに、伸びが止まる数がありません。

長い主系列は、細かく切るほど、本物の二乗和が大きく下がります。偽の星には帯が無いので、そこまでは下がりません。Gapも、k-meansと同じ「中心までの距離」の物差しで測っているのです。


### シルエットの考え

星ごとに、自分の群の中の近さ$a_i$（同じ群の星までの距離の平均）と、となりの群までの近さ$b_i$（いちばん近い別の群の星までの距離の平均）を比べます。

$$
s_i = \frac{b_i - a_i}{\max(a_i, b_i)}
$$

自分の群の中で近く、となりの群から遠いほど、1に近くなります。全部の星の$s_i$を平均し（シルエット）、いちばん大きい群の数を選びます。


### お手本：シルエット

- 役割：群の数を2から8まで替えて、シルエットをリストにためる
- 入力：配列`X`
- 出力：リスト`sils`（7つのシルエット）


In [ ]:
from sklearn.metrics import silhouette_score
sils = []
for k in range(2, 9):
    lab_k = KMeans(k, n_init=10, random_state=0).fit_predict(X)
    sils.append(silhouette_score(X, lab_k, sample_size=5000,
                                 random_state=0))
print(np.round(sils, 3))


#### 1行ずつ読む

- 1行目：シルエットの道具を呼び出す
- 2行目：空のリスト`sils`を用意する
- 3・4行目：群の数を2から8まで替え、当てはめて群の番号を出す（`fit_predict`）
- 5・6行目：シルエットを出してためる。計算が重いので、5,000個を抜き出して測る
- 7行目：7つのシルエットを表示する


In [ ]:
plt.plot(range(2, 9), sils, 'o-', color='tab:green')
plt.xlabel('群の数')
plt.ylabel('シルエット')
plt.show()


3つに分けたときが、いちばん大きくなりました。2節のTODO②で見たとおり、3つに分けると、白色矮星が1つの群になり、主系列は明るい側と暗い側に切られます。


> 重要：群の数の指針は、数を1つ出してくれる。でも、その指針もk-meansと同じ距離の物差しで測っている。最後は、群の中身を読んで決める。


### TODO④：群の数の範囲を替える（予想 → 実行 → 比べる）

下のセルは、群の数を1から8まで替えて、Gapを出します。

まず、予想を書いてください。12まで増やすと、Gapはどこかで伸びが止まるでしょうか。


（ここに予想を書く）


予想を書いたら、セルの`9`を`13`に替えて実行してください（1分ほどかかります）。


In [ ]:
# TODO: 9 を 13 に替えて実行する（少し時間がかかる）
rng = np.random.default_rng(0)
gaps_x = []
for k in range(1, 9):
    fake_w = []
    for b in range(5):
        fake = rng.uniform(X.min(axis=0), X.max(axis=0), size=X.shape)
        fake_w.append(log_w(fake, k))
    gaps_x.append(np.mean(fake_w) - log_w(X, k))
print(np.round(gaps_x, 3))


### やってみよう：予想と比べる

実行した結果を、予想と比べてください。Gapの伸びは、どこかで止まりましたか。


（ここに答えを書く）


12まで増やしても、Gapは伸び続けます。主系列は、いくらでも細かく切れます。Gapの規則では、群の数が決まりません。


### ここまでで分かったこと

| 段 | 分かったこと |
|---|---|
| 星を知る（1節） | 27,860個の星。HR図に主系列と白色矮星（2,027個）が見える |
| 2つに分ける（2節） | k-meansで2つに分けると、白色矮星が1,422個と605個に割れる |
| くり返しを読む（3節） | 割り当てと平均を17回くり返して止まる。二乗和は輪切りが小さい |
| 群の数を決める（4節） | シルエットは3が最大。Gapは8まで伸び続ける |
| 出発点を替える（5節） | ？（このあとの段で） |

まだ分からないのは、出発点を替えても、いつも同じ答えに止まるか、です。


---


## 5. 答えは1つか？


ここから、段5「出発点を替える」（5節）です。問いは、出発点を替えても、k-meansはいつも同じ答えに止まるか、です。

群の数の指針は、シルエットなら3で、Gapは止まりませんでした。3節では、見た目どおりの分け方から始めても、そこで止まりました。出発点しだいで、止まる所は変わるのでしょうか。

k-meansは、出発点から、二乗和が下がる向きにだけ動き、下がらなくなったら止まります。止まった所が、いちばん低いとは限りません。これを局所解といいます。3つに分けるとき、出発点を20通り替えて、止まった二乗和を並べてみます。


### お手本：出発点を20通り試す

- 役割：出発点をでたらめに選び直して20回分け、止まったときの二乗和をためる
- 入力：配列`X`
- 出力：リスト`ss3`（20個の二乗和）


In [ ]:
ss3 = []
for s in range(20):
    km_s = KMeans(3, init='random', n_init=1, random_state=s)
    km_s.fit(X)
    ss3.append(round(km_s.inertia_))
print(ss3)


#### 1行ずつ読む

- 1行目：二乗和をためる、空のリストを作る
- 2行目：乱数のシード`s`を、0から19まで替えてくり返す
- 3行目：群の数3。`init='random'`で星を3つでたらめに選んで出発点にし、`n_init=1`で1通りだけ試す
- 4行目：止まるまでくり返して、当てはめる
- 5行目：止まったときの二乗和を整数に丸めて、リストの後ろに足す
- 6行目：20個の二乗和を表示する


#### 中身を見る：3回だけ回す

同じくり返しを3回だけ回し、1回ごとにリストを表示します。名前は`ss3b`にして、上の`ss3`を壊さないようにします。


In [ ]:
ss3b = []
for s in range(3):
    km_s = KMeans(3, init='random', n_init=1, random_state=s)
    km_s.fit(X)
    ss3b.append(round(km_s.inertia_))
    print(s, ss3b)


1回ごとに、リストの後ろに二乗和が1つ増えます。

いちばん小さい二乗和と、それが何回出たかを数えます。


In [ ]:
print(min(ss3), ss3.count(min(ss3)))


いちばん小さい二乗和14038に届いたのは、20回のうち2回だけです。残りの18回は、16186から16191のあたりで止まりました。

止まった答えの1つ（乱数のシード1）を描きます。


In [ ]:
# @title 図：出発点しだいで止まった答え（中身は読み飛ばしてよい）
km_loc = KMeans(3, init='random', n_init=1, random_state=1)
km_loc.fit(X)
plt.figure(figsize=(7, 5))
plt.scatter(df['色'], df['絶対等級'], s=1, c=km_loc.labels_, cmap='viridis')
plt.gca().invert_yaxis()
plt.xlabel('色（BP−RP）')
plt.ylabel('絶対等級')
plt.show()


止まった答えでは、白色矮星が、主系列の中ほどと同じ群に入ります。TODO②で3つに分けた図と見比べてください。

2節からの`n_init=10`は、出発点を10通り試し、二乗和がいちばん小さい答えを残す設定です。TODO②の3つの群も、こうして選ばれていました。3節の「見た目どおりでも止まる」も、同じ局所解の話でした。


> 重要：k-meansは、止まった所がいちばん良いとは限らない（局所解）。出発点を何通りも試し、二乗和がいちばん小さい答えを選ぶ。


### TODO⑤：群の数を2にする（予想 → 実行 → 比べる）

下のセルは、出発点を20通り替えて分け、出てきた二乗和の種類と、いちばん小さい値が出た回数を表示します。

まず、予想を書いてください。2つに分けると、止まる所は何通りあるでしょうか。


（ここに予想を書く）


予想を書いたら、セルの`3`を`2`に替えて実行してください（`KMeans(3`の`3`）。


In [ ]:
# TODO: 3 を 2 に替えて実行する
ss_x = []
for s in range(20):
    km_s = KMeans(3, init='random', n_init=1, random_state=s)
    km_s.fit(X)
    ss_x.append(round(km_s.inertia_))
print(sorted(set(ss_x)), ss_x.count(min(ss_x)))


### やってみよう：予想と比べる

実行した結果を、予想と比べてください。止まる所は何通りでしたか。


（ここに答えを書く）


2つに分けると、止まる所は2通りです。20回のうち19回は輪切り（3節の24,035とほぼ同じ）、残りの1回は45870で、3節の「見た目どおり」（45,877）に近い所です。出発点しだいで、そこにも止まります。


### ここまでで分かったこと

| 段 | 分かったこと |
|---|---|
| 星を知る（1節） | 27,860個の星。HR図に主系列と白色矮星（2,027個）が見える |
| 2つに分ける（2節） | k-meansで2つに分けると、白色矮星が1,422個と605個に割れる |
| くり返しを読む（3節） | 割り当てと平均を17回くり返して止まる。二乗和は輪切りが小さい |
| 群の数を決める（4節） | シルエットは3が最大。Gapは8まで伸び続ける |
| 出発点を替える（5節） | 出発点を20通り替えると、いちばん小さい二乗和に届くのは2回だけ |

5つの段がそろい、今日の問いに答えられます。


### 答え

k-meansで2つに分けると、主系列と白色矮星には分かれません。白色矮星が真っ二つになります。k-meansは中心までの距離の二乗和を小さくするので、長い主系列を輪切りにします。群の数は、シルエットなら3（白色矮星と、主系列の2つ）で、Gapは止まりません。


まだ言えないこと

- 細長い帯を、帯のまま1つの群にする方法
- 2つの帯の境目にある星を、どちらの群とも言い切らない扱い
- 指針どうしが食い違ったとき、どれを信じるか


### レポートなら、こう書く

今日の解析を、期末レポートの項目に写すと、次のようになります。

- 予想：2つに分ければ、主系列と白色矮星に分かれるだろう
- 解析：色と絶対等級を標準化し、k-meansで2つに分けた
- 驚き：白色矮星が2つの群に割れた。目安の線と比べると、1,422個と605個
- 理由：見た目どおりの分け方は、二乗和が45,877で、輪切り（24,035）の2倍近い
- 群の数：シルエットは3が最大、Gapは12まで止まらない。3つの群の中身を読んで決めた
- 限界：k-meansは丸い群を探す。帯のような群には別の方法が要る


---


## 目標に答えられたか


- 今日の目標は「答えの列が無い星を、似たものどうしでまとめると、星の種類が分かれるか。何個に分ければよいか」でした。HR図と群の図と、二乗和・指針の数で答えられますか？
- 1節で、見かけの等級でなく絶対等級を使ったのはなぜですか？
- 2節で、k-meansの前に標準化したのはなぜですか？
- 3節で、`while`の条件`moved > 0`は、何を確かめていましたか？`for`では書きにくいのはなぜですか？
- 3節で、見た目どおりの分け方でも止まるのに、`KMeans`が輪切りを選んだのはなぜですか？
- 4節で、Gapが伸び続けたのはなぜですか？


---


## 今日の要点


- 今日、持ち帰る1文：k-meansは、近い中心に割り当てて中心を平均へ動かすことを、動かなくなるまでくり返す。距離の二乗和を小さくするので、丸い群を探し、細長い群は輪切りにする
- 重要概念（統計）：k-meansは丸い群を探す。中心までの距離の二乗和を小さくする分け方を探すので、細長い群は輪切りにする
- 重要概念（Python）：while。条件を満たすあいだ、くり返す。何回で終わるかが先に決まらないくり返しに使う
- 距離で分ける前に、列の単位をそろえる（標準化）
- 群の番号は名前にすぎない。群の数の指針も、距離の物差しの上に立つ


---


## 次回へ


今日のk-meansは、丸い群しか探せず、星を1つの群に言い切りました。では、群の形が楕円でもよく、境目の星を「どちらの群か、確率で」言える分け方はあるでしょうか。


---


## 課題（提出）


提出するもの：絵2枚（画像ファイル）。今日の重要概念の、統計とPythonを1枚ずつです。提出フォームから出してください。期限はありません。

コードは書きません。上手な絵である必要はありません。紙に手で描いて写真を撮っても、描画アプリで描いてもかまいません。


### 今日の重要概念（2つ）

- 統計：k-meansは丸い群を探す。中心までの距離の二乗和を小さくする分け方を探すので、細長い群は輪切りにする（3節）
- Python：while。条件を満たすあいだ、くり返す。何回で終わるかが先に決まらないくり返しに使う（3節）


### 今日の2枚（絵にする）

2つの重要概念を、1枚ずつ絵にしてください。その考えが1枚で伝わる絵です。数式やコードは描かなくてかまいません。半年後の自分が見て、何の話だったか思い出せる絵を目指します。


<details><summary>詰まったら</summary>

k-meansは丸い群を探す、なら、たとえば「細長い商店街の売店」を描けます。
売店を2つ置き、客はいちばん近い売店へ行きます。売店は、客の多いところへ少しずつ動きます。すると商店街は、真ん中で2つに切られます。通りの東と西という「見た目の2つ」とは限りません。3節の「見た目どおり」と「輪切り」を並べた図が手本になります。

</details>


<details><summary>詰まったら</summary>

whileなら、たとえば「お風呂にお湯を足す」を描けます。
「ぬるいあいだは、足す」。何杯で終わるかは、やってみるまで分かりません。ちょうどよくなったら止めます。3節の`x`を2倍し続ける例や、二乗和が下がっていく図が手本になります。

</details>


---


## 発展（任意）


### 出発点を賢く選ぶ

5節の`init='random'`は、星をでたらめに選んで出発点にしました。`KMeans`の決まりの選び方（k-means++）は、互いに離れた星を選びやすくします。同じ20通りで、いちばん小さい二乗和に届く回数を比べます。


In [ ]:
ss_pp = []
for s in range(20):
    km_s = KMeans(3, n_init=1, random_state=s)
    km_s.fit(X)
    ss_pp.append(round(km_s.inertia_))
print(min(ss_pp), ss_pp.count(min(ss_pp)))


#### 1行ずつ読む

- 3行目：`init`を書かないと、k-means++で出発点を選ぶ
- ほかの行は、5節のお手本と同じ

20回のうち11回が、いちばん小さい二乗和14038に届きました。でたらめ（2回）より、ずっと多くなります。それでも全部ではないので、`n_init`で何通りも試します。

試すなら、群の数を4や5にして、届く回数を比べてみましょう。
